## AND 게이트 구현

입력이 모두 1 일 때만 1 출력

In [9]:
import torch

def AND(x1, x2) :
  # 전달받은 입력값들을 텐서로 생성
  x = torch.tensor([x1, x2], dtype=torch.float32)

  # 가중치와 편향
  w = torch.tensor([0.5, 0.5])

  # 편향 (bias, 기본값 | 시작값) 임의 지정
  b = -0.7
  # y = w1x1 + w2x2 + b

  tmp = torch.sum(w * x) + b
  if tmp <= 0 :
    return 0
  else :
    return 1

print(AND(0, 0))
print(AND(1, 0))
print(AND(0, 1))
print(AND(1, 1))

0
0
0
1


In [11]:
import torch

def NAND(x1, x2) :
  # 전달받은 입력값들을 텐서로 생성
  x = torch.tensor([x1, x2], dtype=torch.float32)

  # 가중치와 편향
  w = torch.tensor([-0.2, -0.2])

  # 편향 (bias, 기본값 | 시작값) 임의 지정
  b = 0.3
  # y = w1x1 + w2x2 + b

  tmp = torch.sum(w * x) + b
  if tmp <= 0 :
    return 0
  else :
    return 1

print(NAND(0, 0))
print(NAND(1, 0))
print(NAND(0, 1))
print(NAND(1, 1))

1
1
1
0


## or 게이트 구현

하나라도 1 이면 1 출력

In [13]:
import torch

def OR(x1, x2) :
  # 전달받은 입력값들을 텐서로 생성
  x = torch.tensor([x1, x2], dtype=torch.float32)

  # 가중치와 편향
  w = torch.tensor([0.7, 0.7])

  # 편향 (bias, 기본값 | 시작값) 임의 지정
  b = -0.3
  # y = w1x1 + w2x2 + b

  tmp = torch.sum(w * x) + b
  if tmp <= 0 :
    return 0
  else :
    return 1

print(OR(0, 0))
print(OR(1, 0))
print(OR(0, 1))
print(OR(1, 1))

0
1
1
1


## 학습없이 Weight 탐색

논리회로를 만족하는 Weight 찾기

In [20]:
import torch

def check(x1, x2, y) :
    while True :
        cnt = 0

        w1 = torch.randn(1).item()
        w2 = torch.randn(1).item()
        b = torch.randn(1).item()

        for idx in range(len(x1)) :
            x = torch.tensor([x1[idx], x2[idx]], dtype=torch.float32)
            w = torch.tensor([w1, w2], dtype=torch.float32)

            # ★ [여기서부터 4칸씩 안으로 밀어 넣었습니다!]
            # 데이터를 꺼낼 때마다 바로바로 계산하고 채점합니다.
            tmp = torch.sum(x*w) + b
            if tmp <= 0:
                res = 0
            else:
                res = 1

            if y[idx] == res :
                cnt += 1
        #for 끝나는 지점 -----------

        # 4개 데이터 모두 정답인지 확인하는 판정대는 for문 '밖', while문 '안'에 둡니다.
        if cnt == len(x1): # 4 대신 len(x1)을 쓰면 더 안전합니다.
            print('★ 만족하는 가중치 조합 발견! ★')
            print('w1 == ', w1)
            print('w2 == ', w2)
            print('b == ', b)
            break

# [테스트용 데이터 정의 - AND 게이트]
x1 = [0, 0, 1, 1]
x2 = [0, 1, 0, 1]
y  = [0, 0, 0, 1]

# 실행해보면 1초도 안 되어서 툭 하고 끝날 겁니다!
check(x1, x2, y)

★ 만족하는 가중치 조합 발견! ★
w1 ==  0.49600809812545776
w2 ==  0.24422325193881989
b ==  -0.6648870706558228


## and 게이트용 weight 탐색

In [21]:
def AND_SEARCH() :
  x1 = [0, 1, 0, 1]
  x2 = [0, 0, 1, 1]
  y = [0,0,0,1]
  check(x1, x2, y)

print('AND')
AND_SEARCH()

AND
★ 만족하는 가중치 조합 발견! ★
w1 ==  0.3650202751159668
w2 ==  1.4573880434036255
b ==  -1.4640555381774902


## nand 게이트의 weight 탐색

In [25]:
def NAND_SEARCH() :
  x1 = [0, 1, 0, 1]
  x2 = [0, 0, 1, 1]
  y = [1,1,1,0]
  check(x1, x2, y)

print('NAND')
NAND_SEARCH()

NAND
★ 만족하는 가중치 조합 발견! ★
w1 ==  -0.4642331898212433
w2 ==  -0.9317575693130493
b ==  1.04238760471344


## or 게이트 weight 탐색

In [26]:
def OR_SEARCH() :
  x1 = [0, 1, 0, 1]
  x2 = [0, 0, 1, 1]
  y = [0,1,1,1]
  check(x1, x2, y)

print('OR')
OR_SEARCH()

OR
★ 만족하는 가중치 조합 발견! ★
w1 ==  0.9362375140190125
w2 ==  0.8872539401054382
b ==  -0.7285195589065552


## 단일 퍼셉트론 TORCH 구현



In [35]:
import torch
import torch.nn as nn
import torch.optim as optim

# 단일 퍼셉트론
model = nn.Linear(2, 1)

# criterion = nn.MSELoss()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.01)

X = torch.tensor([
    [0., 1.],
    [0., 1.],
    [1., 0.],
    [1., 1.]])

y = torch.tensor([
    [0.],
    [0.],
    [0.],
    [1.],
])

for epoch in range(1000):
  pred = model(X)
  loss = criterion(pred, y)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoch % 100 == 0:
    print(f'epoch: {epoch}, loss: {loss.item()}')

print(model(X))

epoch: 0, loss: 0.9440338611602783
epoch: 100, loss: 0.6065303087234497
epoch: 200, loss: 0.517903745174408
epoch: 300, loss: 0.4536193013191223
epoch: 400, loss: 0.39438003301620483
epoch: 500, loss: 0.341552197933197
epoch: 600, loss: 0.29585930705070496
epoch: 700, loss: 0.25701791048049927
epoch: 800, loss: 0.2242819368839264
epoch: 900, loss: 0.19676944613456726
tensor([[-2.3171],
        [-2.3171],
        [-1.2809],
        [ 1.2082]], grad_fn=<AddmmBackward0>)


## XOR 게이트 만들기


In [39]:
def XOR(x1, x2) :
    s1 = NAND(x1, x2)
    s2 = OR(x1, x2)
    y = AND(s1, s2)
    return y

input_data = [  [0,0],
                [1,0],
                [0,1],
                [1,1] ]

print('XOR')
for idx in input_data:
    print(XOR(idx[0], idx[1]))

XOR
0
1
1
0


## 가장 기본적인 다중퍼셉트론 구조

In [40]:
import torch
import torch.nn as nn

class SimpleDNN(nn.Module):
    def __init__(self):
        super().__init__()

        # [1층 입력층 -> 은닉층1] 10개의 특징을 받아 64개의 파생 조건 노드로 확장
        self.fc1 = nn.Linear(10, 64)

        # [2층 은닉층1 -> 은닉층2] 64개의 파생 조건을 조합해 32개의 심화 파생 조건으로 압축
        self.fc2 = nn.Linear(64, 32)

        # [3층 은닉층2 -> 출력층] 32개의 심화 파생 조건을 모아 최종 1개의 예측값 도출
        self.fc3 = nn.Linear(32, 1)

        # 활성화 함수 (Dead ReLU 위험이 있지만 가장 대중적인 기본 필터)
        self.relu = nn.ReLU()

    def forward(self, x):
        # 1층 연산: (X × W1) + b1 한 뒤 ReLU로 음수 커트라인 적용
        x = self.relu(self.fc1(x))

        # 2층 연산: (1층결과 × W2) + b2 한 뒤 ReLU로 또 음수 커트라인 적용
        x = self.relu(self.fc2(x))

        # 3층 연산: 최종 출력층은 확률이나 점수 원본을 내야 하므로 ReLU를 거치지 않음
        x = self.fc3(x)

        return x

# 모델 객체 생성 및 구조 출력
model = SimpleDNN()
print(model)

SimpleDNN(
  (fc1): Linear(in_features=10, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)
